# Rag 

R = Retrieval
    관련 문서 검색

A = Augmentation
    검색된 문서를 질문과 함께 프롬프트에 넣음

G = Generation
    LLM이 문서를 바탕으로 답변 생성

## 문서 확인

In [5]:
import json

In [6]:
file_path = "../data/KorQuAD_v1.0_train.json"

with open(file_path, "r", encoding="utf-8") as file:
    korquad = json.load(file)


print(type(korquad))
print(korquad.keys())

<class 'dict'>
dict_keys(['version', 'data'])


In [7]:
articles = korquad["data"]

print(type(articles))
print("전체 문서 수:", len(articles))

<class 'list'>
전체 문서 수: 1420


In [8]:
article = articles[0]

print(article.keys())
print("제목:", article["title"])

dict_keys(['paragraphs', 'title'])
제목: 파우스트_서곡


In [9]:
paragraphs = article["paragraphs"]

print("문단 수:", len(paragraphs))
print(paragraphs[0].keys())

문단 수: 3
dict_keys(['qas', 'context'])


In [10]:
paragraph = paragraphs[0]

print("본문:")
print(paragraph["context"])

본문:
1839년 바그너는 괴테의 파우스트을 처음 읽고 그 내용에 마음이 끌려 이를 소재로 해서 하나의 교향곡을 쓰려는 뜻을 갖는다. 이 시기 바그너는 1838년에 빛 독촉으로 산전수전을 다 걲은 상황이라 좌절과 실망에 가득했으며 메피스토펠레스를 만나는 파우스트의 심경에 공감했다고 한다. 또한 파리에서 아브네크의 지휘로 파리 음악원 관현악단이 연주하는 베토벤의 교향곡 9번을 듣고 깊은 감명을 받았는데, 이것이 이듬해 1월에 파우스트의 서곡으로 쓰여진 이 작품에 조금이라도 영향을 끼쳤으리라는 것은 의심할 여지가 없다. 여기의 라단조 조성의 경우에도 그의 전기에 적혀 있는 것처럼 단순한 정신적 피로나 실의가 반영된 것이 아니라 베토벤의 합창교향곡 조성의 영향을 받은 것을 볼 수 있다. 그렇게 교향곡 작곡을 1839년부터 40년에 걸쳐 파리에서 착수했으나 1악장을 쓴 뒤에 중단했다. 또한 작품의 완성과 동시에 그는 이 서곡(1악장)을 파리 음악원의 연주회에서 연주할 파트보까지 준비하였으나, 실제로는 이루어지지는 않았다. 결국 초연은 4년 반이 지난 후에 드레스덴에서 연주되었고 재연도 이루어졌지만, 이후에 그대로 방치되고 말았다. 그 사이에 그는 리엔치와 방황하는 네덜란드인을 완성하고 탄호이저에도 착수하는 등 분주한 시간을 보냈는데, 그런 바쁜 생활이 이 곡을 잊게 한 것이 아닌가 하는 의견도 있다.


In [11]:
qas = paragraph["qas"]

print("질문 개수:", len(qas))
print(qas[0].keys())

질문 개수: 8
dict_keys(['answers', 'id', 'question'])


In [12]:
qa = qas[0]

print("질문:", qa["question"])
print("정답:", qa["answers"])

질문: 바그너는 괴테의 파우스트를 읽고 무엇을 쓰고자 했는가?
정답: [{'text': '교향곡', 'answer_start': 54}]


In [13]:
answer = qa["answers"][0]

print("정답:", answer["text"])
print("정답 시작 위치:", answer["answer_start"])

정답: 교향곡
정답 시작 위치: 54


In [14]:
context = paragraph["context"]
answer_text = answer["text"]
answer_start = answer["answer_start"]

print("정답:", answer_text)
print("정답 시작 위치:", answer_start)
print("본문에서 찾은 정답:", context[answer_start:answer_start + len(answer_text)])

정답: 교향곡
정답 시작 위치: 54
본문에서 찾은 정답: 교향곡


In [15]:
print("정답 위치 일치 여부:", answer_text == context[answer_start:answer_start + len(answer_text)])

정답 위치 일치 여부: True


## KorQuAD 샘플 만들기 (rag형식에 맞게 데이터 가공)

In [16]:
documents = []
questions = []

max_documents = 10

In [17]:
for article in articles:
    for paragraph in article["paragraphs"]:
        document_id = f"doc_{len(documents)}"

        documents.append({
            "document_id": document_id,
            "title": article["title"],
            "text": paragraph["context"]      # 문단마다 context를 document로 취급
        })

        for qa in paragraph["qas"]:
            answer = qa["answers"][0]

            questions.append({
                "question_id": qa["id"],
                "question": qa["question"],
                "gold_answer": answer["text"],
                "answer_start": answer["answer_start"],
                "document_id": document_id
            })

        if len(documents) == max_documents:
            break

    if len(documents) == max_documents:
        break

In [18]:
print("문서 수:", len(documents))
print("질문 수:", len(questions))

print(documents[0].keys())
print(questions[0].keys())

문서 수: 10
질문 수: 70
dict_keys(['document_id', 'title', 'text'])
dict_keys(['question_id', 'question', 'gold_answer', 'answer_start', 'document_id'])


In [19]:
print("문서 수:", len(documents))
print("질문 수:", len(questions))

print("\n첫 번째 문서")
print("ID:", documents[0]["document_id"])
print("제목:", documents[0]["title"])
print("본문:", documents[0]["text"][:200])

print("\n첫 번째 질문")
print("질문 ID:", questions[0]["question_id"])
print("질문:", questions[0]["question"])
print("정답:", questions[0]["gold_answer"])
print("연결 문서 ID:", questions[0]["document_id"])

문서 수: 10
질문 수: 70

첫 번째 문서
ID: doc_0
제목: 파우스트_서곡
본문: 1839년 바그너는 괴테의 파우스트을 처음 읽고 그 내용에 마음이 끌려 이를 소재로 해서 하나의 교향곡을 쓰려는 뜻을 갖는다. 이 시기 바그너는 1838년에 빛 독촉으로 산전수전을 다 걲은 상황이라 좌절과 실망에 가득했으며 메피스토펠레스를 만나는 파우스트의 심경에 공감했다고 한다. 또한 파리에서 아브네크의 지휘로 파리 음악원 관현악단이 연주하는 베토벤의 교

첫 번째 질문
질문 ID: 6566495-0-0
질문: 바그너는 괴테의 파우스트를 읽고 무엇을 쓰고자 했는가?
정답: 교향곡
연결 문서 ID: doc_0


## Chunking (검색하기 좋은 작은 조각으로 나누기)

In [20]:
for document in documents:
    print(
        document["document_id"],
        document["title"],
        len(document["text"])
    )

doc_0 파우스트_서곡 673
doc_1 파우스트_서곡 504
doc_2 파우스트_서곡 447
doc_3 커닐링구스 417
doc_4 커닐링구스 416
doc_5 커닐링구스 475
doc_6 한와_선 454
doc_7 한와_선 404
doc_8 한와_선 392
doc_9 한와_선 469


In [21]:
def split_text(text, chunk_size=300, overlap=50):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)

        start = end - overlap

    return chunks

In [22]:
test_chunks = split_text(documents[0]["text"])

print("원본 길이:", len(documents[0]["text"]))
print("Chunk 개수:", len(test_chunks))

for index, chunk in enumerate(test_chunks):
    print(f"\nChunk {index}")
    print("길이:", len(chunk))
    print(chunk)

원본 길이: 673
Chunk 개수: 3

Chunk 0
길이: 300
1839년 바그너는 괴테의 파우스트을 처음 읽고 그 내용에 마음이 끌려 이를 소재로 해서 하나의 교향곡을 쓰려는 뜻을 갖는다. 이 시기 바그너는 1838년에 빛 독촉으로 산전수전을 다 걲은 상황이라 좌절과 실망에 가득했으며 메피스토펠레스를 만나는 파우스트의 심경에 공감했다고 한다. 또한 파리에서 아브네크의 지휘로 파리 음악원 관현악단이 연주하는 베토벤의 교향곡 9번을 듣고 깊은 감명을 받았는데, 이것이 이듬해 1월에 파우스트의 서곡으로 쓰여진 이 작품에 조금이라도 영향을 끼쳤으리라는 것은 의심할 여지가 없다. 여기의 라단조 조성의 

Chunk 1
길이: 300
이 작품에 조금이라도 영향을 끼쳤으리라는 것은 의심할 여지가 없다. 여기의 라단조 조성의 경우에도 그의 전기에 적혀 있는 것처럼 단순한 정신적 피로나 실의가 반영된 것이 아니라 베토벤의 합창교향곡 조성의 영향을 받은 것을 볼 수 있다. 그렇게 교향곡 작곡을 1839년부터 40년에 걸쳐 파리에서 착수했으나 1악장을 쓴 뒤에 중단했다. 또한 작품의 완성과 동시에 그는 이 서곡(1악장)을 파리 음악원의 연주회에서 연주할 파트보까지 준비하였으나, 실제로는 이루어지지는 않았다. 결국 초연은 4년 반이 지난 후에 드레스덴에서 연주되었고 재연도

Chunk 2
길이: 173
로는 이루어지지는 않았다. 결국 초연은 4년 반이 지난 후에 드레스덴에서 연주되었고 재연도 이루어졌지만, 이후에 그대로 방치되고 말았다. 그 사이에 그는 리엔치와 방황하는 네덜란드인을 완성하고 탄호이저에도 착수하는 등 분주한 시간을 보냈는데, 그런 바쁜 생활이 이 곡을 잊게 한 것이 아닌가 하는 의견도 있다.


In [23]:
all_chunks = []

for document in documents:
    split_chunks = split_text(document["text"])

    for chunk_index, chunk_text in enumerate(split_chunks):
        all_chunks.append({
            "chunk_id": f'{document["document_id"]}_chunk_{chunk_index}',
            "document_id": document["document_id"],
            "title": document["title"],
            "chunk_index": chunk_index,
            "text": chunk_text
        })

In [24]:
print("Chunk ID:", all_chunks[0]["chunk_id"])
print("원본 문서 ID:", all_chunks[0]["document_id"])
print("제목:", all_chunks[0]["title"])
print("Chunk 길이:", len(all_chunks[0]["text"]))
print("내용:", all_chunks[0]["text"][:200])

Chunk ID: doc_0_chunk_0
원본 문서 ID: doc_0
제목: 파우스트_서곡
Chunk 길이: 300
내용: 1839년 바그너는 괴테의 파우스트을 처음 읽고 그 내용에 마음이 끌려 이를 소재로 해서 하나의 교향곡을 쓰려는 뜻을 갖는다. 이 시기 바그너는 1838년에 빛 독촉으로 산전수전을 다 걲은 상황이라 좌절과 실망에 가득했으며 메피스토펠레스를 만나는 파우스트의 심경에 공감했다고 한다. 또한 파리에서 아브네크의 지휘로 파리 음악원 관현악단이 연주하는 베토벤의 교


## Embedding

In [25]:
# %pip install sentence-transformers

In [26]:
from sentence_transformers import SentenceTransformer 
# 임베딩 모델을 사용할 도구 가져옴

model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
# Hugging Face에 있는 모델의 주소 지정

embedding_model = SentenceTransformer(model_name)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1180.67it/s]


In [29]:
chunk_texts = [chunk["text"] for chunk in all_chunks]

print("임베딩할 Chunk 수:", len(chunk_texts))

임베딩할 Chunk 수: 22


In [30]:
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=16,               # 한 번에 처리할 Chunk 수
    show_progress_bar=True,      # 진행률 표시
    convert_to_numpy=True,       # 결과를 NumPy 배열로 반환
    normalize_embeddings=True    # 벡터 길이 맞춰 유사도 비교하기 편하게 만듦
)

Batches: 100%|██████████| 2/2 [00:16<00:00,  8.32s/it]


## Chroma 저장 (vector DB)

In [31]:
print(all_chunks[0].keys())
print("Chunk 수:", len(all_chunks))
print("임베딩 크기:", chunk_embeddings.shape)

assert len(all_chunks) == len(chunk_embeddings)

dict_keys(['chunk_id', 'document_id', 'title', 'chunk_index', 'text'])
Chunk 수: 22
임베딩 크기: (22, 384)


In [31]:
# %pip install chromadb

In [32]:
import chromadb


chroma_client = chromadb.PersistentClient(
    path="vector_db"
)
# Chroma DB 만들고 조회할 수 있는 관리 도구 생성


collection = chroma_client.get_or_create_collection(
    name="korquad_practice"
)
# Chroma DB 안에 KorQuAD 데이터를 저장할 컬렉션 생성 (임베딩 벡터와 Chunk 텍스트를 묶어서 저장)
# 컬렉션 : DB 안의 데이터 묶음 (chuck0, chunk1, chunk2, ...)

### Chroma에 넣을 값 분리하기

In [33]:
chunk_ids = [chunk["chunk_id"] for chunk in all_chunks]  # 각 chuck의 고유 ID
chunk_texts = [chunk["text"] for chunk in all_chunks]   # 실제 검색할 본문

In [34]:
# metadata : 임베딩 벡터와 함께 저장할 추가 정보 (문서 ID, 제목, Chunk 인덱스 등)

chunk_metadatas = [
    {
        "document_id": chunk["document_id"],
        "title": chunk["title"],
        "chunk_index": chunk["chunk_index"]
    }
    for chunk in all_chunks
]

In [35]:
print(len(chunk_ids))
print(len(chunk_texts))
print(len(chunk_metadatas))
print(len(chunk_embeddings))

22
22
22
22


### Chroma에 저장하기

In [36]:
collection.upsert(
    ids=chunk_ids,
    documents=chunk_texts,
    metadatas=chunk_metadatas,
    embeddings=chunk_embeddings.tolist()
)

chunk_embeddings.tolist()

[[-0.0033454576041549444,
  0.11756972968578339,
  -0.04052267596125603,
  0.027984173968434334,
  -0.07879999279975891,
  0.05779549852013588,
  0.10100245475769043,
  0.0031276713125407696,
  0.02790926769375801,
  -0.07152757048606873,
  -0.011952729895710945,
  -0.0008425339474342763,
  0.021825483068823814,
  -0.06365110725164413,
  0.016558334231376648,
  -0.026609547436237335,
  0.0217566080391407,
  0.060826465487480164,
  -0.011899602599442005,
  -0.03119421750307083,
  -0.017613407224416733,
  0.007757027633488178,
  0.008768091909587383,
  -0.024125823751091957,
  -0.02669908106327057,
  -0.05098242312669754,
  0.032189249992370605,
  -0.04536713287234306,
  -0.014780744910240173,
  0.0001986053102882579,
  -0.009752938523888588,
  0.002755122259259224,
  0.07314549386501312,
  -0.028841646388173103,
  -0.015614154748618603,
  0.022938037291169167,
  0.019419770687818527,
  0.03496275842189789,
  0.021540889516472816,
  0.03169282525777817,
  0.06550969183444977,
  0.0909286

In [37]:
print("저장된 Chunk 수:", collection.count())

저장된 Chunk 수: 22


## Retrieval (관련 chuck 검색; Top-k)

In [38]:
# 질문 임베딩

test_question = questions[0]["question"]


question_embedding = embedding_model.encode(
    test_question,
    normalize_embeddings=True                 # 벡터 길이 맞춰 유사도 비교하기 편하게 만듦 (위에서 384로 맞춰짐)
)

print("질문 임베딩 크기:", question_embedding.shape)

질문 임베딩 크기: (384,)


### top-3

In [50]:
# Chroma에서 Top-3 검색

search_results_3 = collection.query(
    query_embeddings=[question_embedding.tolist()],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

documents
- 검색된 Chunk의 실제 본문 

metadatas
- 제목, 원본 문서 ID, Chunk 순서

distances
- 질문 벡터와 해당 Chunk 벡터 사이의 거리

In [51]:
print(type(search_results_3))
print(search_results_3.keys())

<class 'dict'>
dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])


In [52]:
for rank in range(3):
    print(f"\n검색 순위 {rank + 1}")
    print("Chunk ID:", search_results_3["ids"][0][rank])
    print("거리:", search_results_3["distances"][0][rank])
    print("메타데이터:", search_results_3["metadatas"][0][rank])
    print("본문:", search_results_3["documents"][0][rank])


검색 순위 1
Chunk ID: doc_1_chunk_1
거리: 0.8035519123077393
메타데이터: {'chunk_index': 1, 'document_id': 'doc_1', 'title': '파우스트_서곡'}
본문: 에 리스트가 자신의 작품 파우스트 교향곡을 거의 완성하여 그 사실을 바그너에게 알렸고, 바그너는 다시 개정된 총보를 리스트에게 보내고 브라이트코프흐 & 헤르텔 출판사에는 20루이의 금을 받고 팔았다. 또한 그의 작품을 “하나하나의 음표가 시인의 피로 쓰여졌다”며 극찬했던 한스 폰 뷜로가 그것을 피아노 독주용으로 편곡했는데, 리스트는 그것을 약간 변형되었을 뿐이라고 지적했다. 이 서곡의 총보 첫머리에는 파우스트 1부의 내용 중 한 구절을 인용하고 있다.

검색 순위 2
Chunk ID: doc_0_chunk_0
거리: 1.0625693798065186
메타데이터: {'title': '파우스트_서곡', 'document_id': 'doc_0', 'chunk_index': 0}
본문: 1839년 바그너는 괴테의 파우스트을 처음 읽고 그 내용에 마음이 끌려 이를 소재로 해서 하나의 교향곡을 쓰려는 뜻을 갖는다. 이 시기 바그너는 1838년에 빛 독촉으로 산전수전을 다 걲은 상황이라 좌절과 실망에 가득했으며 메피스토펠레스를 만나는 파우스트의 심경에 공감했다고 한다. 또한 파리에서 아브네크의 지휘로 파리 음악원 관현악단이 연주하는 베토벤의 교향곡 9번을 듣고 깊은 감명을 받았는데, 이것이 이듬해 1월에 파우스트의 서곡으로 쓰여진 이 작품에 조금이라도 영향을 끼쳤으리라는 것은 의심할 여지가 없다. 여기의 라단조 조성의 

검색 순위 3
Chunk ID: doc_1_chunk_0
거리: 1.0839712619781494
메타데이터: {'chunk_index': 0, 'title': '파우스트_서곡', 'document_id': 'doc_1'}
본문: 한편 1840년부터 바그너와 알고 지내던 리스트가 잊혀져 있던 1악장을 부활시켜 1852년에 바이

In [53]:
print("정답 문서 ID:", questions[0]["document_id"])
print("1위 검색 문서 ID:", search_results_3["metadatas"][0][0]["document_id"])

정답 문서 ID: doc_0
1위 검색 문서 ID: doc_1


Top_1 검색 실패했지만 Top_3 검색은 성공

### Top-5

In [54]:
# Chroma에서 Top-5 검색

search_results_5 = collection.query(
    query_embeddings=[question_embedding.tolist()],
    n_results=5,
    include=["documents", "metadatas", "distances"]
)

In [55]:
print(type(search_results_5))
print(search_results_5.keys())

<class 'dict'>
dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])


In [ ]:
for rank in range(5):
    print(f"\n검색 순위 {rank + 1}")
    print("Chunk ID:", search_results_5["ids"][0][rank])
    print("거리:", search_results_5["distances"][0][rank])
    print("메타데이터:", search_results_5["metadatas"][0][rank])
    print("본문:", search_results_5["documents"][0][rank])


검색 순위 1
Chunk ID: doc_1_chunk_1
거리: 0.8035519123077393
메타데이터: {'chunk_index': 1, 'title': '파우스트_서곡', 'document_id': 'doc_1'}
본문: 에 리스트가 자신의 작품 파우스트 교향곡을 거의 완성하여 그 사실을 바그너에게 알렸고, 바그너는 다시 개정된 총보를 리스트에게 보내고 브라이트코프흐 & 헤르텔 출판사에는 20루이의 금을 받고 팔았다. 또한 그의 작품을 “하나하나의 음표가 시인의 피로 쓰여졌다”며 극찬했던 한스 폰 뷜로가 그것을 피아노 독주용으로 편곡했는데, 리스트는 그것을 약간 변형되었을 뿐이라고 지적했다. 이 서곡의 총보 첫머리에는 파우스트 1부의 내용 중 한 구절을 인용하고 있다.

검색 순위 2
Chunk ID: doc_0_chunk_0
거리: 1.0625693798065186
메타데이터: {'chunk_index': 0, 'title': '파우스트_서곡', 'document_id': 'doc_0'}
본문: 1839년 바그너는 괴테의 파우스트을 처음 읽고 그 내용에 마음이 끌려 이를 소재로 해서 하나의 교향곡을 쓰려는 뜻을 갖는다. 이 시기 바그너는 1838년에 빛 독촉으로 산전수전을 다 걲은 상황이라 좌절과 실망에 가득했으며 메피스토펠레스를 만나는 파우스트의 심경에 공감했다고 한다. 또한 파리에서 아브네크의 지휘로 파리 음악원 관현악단이 연주하는 베토벤의 교향곡 9번을 듣고 깊은 감명을 받았는데, 이것이 이듬해 1월에 파우스트의 서곡으로 쓰여진 이 작품에 조금이라도 영향을 끼쳤으리라는 것은 의심할 여지가 없다. 여기의 라단조 조성의 

검색 순위 3
Chunk ID: doc_1_chunk_0
거리: 1.0839712619781494
메타데이터: {'document_id': 'doc_1', 'chunk_index': 0, 'title': '파우스트_서곡'}
본문: 한편 1840년부터 바그너와 알고 지내던 리스트가 잊혀져 있던 1악장을 부활시켜 1852년에 바이

In [ ]:
print("정답 문서 ID:", questions[0]["document_id"])
print("1위 검색 문서 ID:", search_results_5["metadatas"][0][0]["document_id"])

정답 문서 ID: doc_0
1위 검색 문서 ID: doc_1


top-3와 결과 같음

top-5는 관련 없는 chunk(doc_5)까지 LLM에 들어감
- 모델이 헷갈릴 수 있음 -> 입력 문맥이 길어짐 -> 생성 속도와 비용 증가

따라서 top-3 채택

In [ ]:
# 검색된 Chunk 3개를 LLM에 전달할 하나의 문맥 문자열로 합치기

retrieved_context = "\n\n".join(
    search_results_3["documents"][0]
)

## Generation  (검색된 chunk를 근거로 답변 생성-LLM)

#### 1) Hugging Face "Qwen/Qwen2.5-0.5B-Instruct" 

- Chunk와 질문을 함께 읽고 가장 적절할 가능성이 높은 답변 문장을 한 글자·단어씩 생성하는 방식 (사전학습 후 지시를 따르도록 추가 학습한 모델)

In [ ]:
# %pip install -U transformers accelerate

: 

In [1]:
from transformers import pipeline


generation_model_name = "Qwen/Qwen2.5-0.5B-Instruct"


# 모델 불러오기
generator = pipeline(
    task="text-generation",
    model=generation_model_name,
    dtype="auto"                     # PC 환경에 맞는 숫자 정밀도를 자동 선택
)

c:\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1169.78it/s]


task="text-generation"       # 이어지는 글 생성

task="text-classification"   # 텍스트 분류

task="question-answering"    # 주어진 문맥에서 답 찾기

task="summarization"         # 요약

task="translation"           # 번역

In [42]:
# 모델에 전달할 메시지 만들기 (채팅형 모델에 알맞은 형식)

messages = [
    {
        "role": "system",
        "content": (
            "검색된 문맥만 사용해 질문에 답하세요. "
            "문맥에 답이 없으면 '문맥에서 답을 찾을 수 없습니다.'라고 답하세요. "
            "답변은 간결한 한 문장으로 작성하세요."
        )
    },
    {
        "role": "user",
        "content": f"""
[검색된 문맥]
{retrieved_context}

[질문]
{test_question}
"""
    }
]


print(messages)

[{'role': 'system', 'content': "검색된 문맥만 사용해 질문에 답하세요. 문맥에 답이 없으면 '문맥에서 답을 찾을 수 없습니다.'라고 답하세요. 답변은 간결한 한 문장으로 작성하세요."}, {'role': 'user', 'content': '\n[검색된 문맥]\n에 리스트가 자신의 작품 파우스트 교향곡을 거의 완성하여 그 사실을 바그너에게 알렸고, 바그너는 다시 개정된 총보를 리스트에게 보내고 브라이트코프흐 & 헤르텔 출판사에는 20루이의 금을 받고 팔았다. 또한 그의 작품을 “하나하나의 음표가 시인의 피로 쓰여졌다”며 극찬했던 한스 폰 뷜로가 그것을 피아노 독주용으로 편곡했는데, 리스트는 그것을 약간 변형되었을 뿐이라고 지적했다. 이 서곡의 총보 첫머리에는 파우스트 1부의 내용 중 한 구절을 인용하고 있다.\n\n1839년 바그너는 괴테의 파우스트을 처음 읽고 그 내용에 마음이 끌려 이를 소재로 해서 하나의 교향곡을 쓰려는 뜻을 갖는다. 이 시기 바그너는 1838년에 빛 독촉으로 산전수전을 다 걲은 상황이라 좌절과 실망에 가득했으며 메피스토펠레스를 만나는 파우스트의 심경에 공감했다고 한다. 또한 파리에서 아브네크의 지휘로 파리 음악원 관현악단이 연주하는 베토벤의 교향곡 9번을 듣고 깊은 감명을 받았는데, 이것이 이듬해 1월에 파우스트의 서곡으로 쓰여진 이 작품에 조금이라도 영향을 끼쳤으리라는 것은 의심할 여지가 없다. 여기의 라단조 조성의 \n\n한편 1840년부터 바그너와 알고 지내던 리스트가 잊혀져 있던 1악장을 부활시켜 1852년에 바이마르에서 연주했다. 이것을 계기로 바그너도 이 작품에 다시 관심을 갖게 되었고, 그 해 9월에는 총보의 반환을 요구하여 이를 서곡으로 간추린 다음 수정을 했고 브라이트코프흐 & 헤르텔 출판사에서 출판할 개정판도 준비했다. 1853년 5월에는 리스트가 이 작품이 수정되었다는 것을 인정했지만, 끝내 바그너의 출판 계획은 무산되고 말았다. 이후 1855년에 리스트가 자신의 작품 파우스트 교향곡을 거의 완

system
→ 모델이 전체적으로 따라야 할 규칙

user
→ 사용자의 실제 질문

assistant
→ 모델이 이전에 했던 답변

In [43]:
# Qwen 모델에 메시지 전달하고 답변 생성 (실질적인 generation)

outputs = generator(
    messages,
    max_new_tokens=50,
    do_sample=False       # 무작위로 고르지 않고 가장 가능성 높은 토큰을 선택
)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


KeyboardInterrupt: 

속도가 너무 느림.... 그래서 일단 성능을 보려고 밑의 간단한 질문으로 다시 물어봤음

In [45]:
short_messages = [
    {
        "role": "user",
        "content": "대한민국의 수도를 한 단어로 답하세요."
    }
]


import time

start_time = time.perf_counter()

test_outputs = generator(
    short_messages,
    max_new_tokens=10,
    do_sample=False
)

print(f"생성 시간: {time.perf_counter() - start_time:.2f}초")
print(test_outputs)

[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


생성 시간: 131.04초
[{'generated_text': [{'role': 'user', 'content': '대한민국의 수도를 한 단어로 답하세요.'}, {'role': 'assistant', 'content': '首尔 (Hong Kong)'}]}]


- 가볍다는 이유로 Qwen2.5-0.5B-Instruct를 골랐는데, 내 CPU 환경에서는 속도도 부족하고 한국어 질의응답 품질도 신뢰하기 어려운 상태

#### 2) Hugging Face "naver-hyperclovax/HyperCLOVAX-SEED-Text-Instruct-1.5B"

- 네이버가 공개한 공식 모델
- Text Generation 모델 (입력 문장 뒤 이어질 토큰 생성)
- AutoModelForCausalLM으로 실행하는 생성형 모델 (생성형 언어 모델을 자동으로 불러오는 도구 클래스)
- 한국어와 한국 문화 이해를 중심으로 개발
- 1.5B 파라미터
- 최대 16K 토큰 문맥 지원

cf) CausalLM : 앞의 토큰을 보고 다음 토큰 예측

In [2]:
from transformers import AutoTokenizer

model_name = (
    "naver-hyperclovax/"
    "HyperCLOVAX-SEED-Text-Instruct-1.5B"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

print(type(tokenizer))
print("모델 접근 성공")

c:\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Python312\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yejib\.cache\huggingface\hub\models--naver-hyperclovax--HyperCLOVAX-SEED-Text-Instruct-1.5B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, se

<class 'transformers.models.gpt2.tokenization_gpt2.GPT2Tokenizer'>
모델 접근 성공
